# 🧼 SYSTÈME DE COMPTAGE AUTOMATIQUE DE SAVONS
## Huileries Belhassane - Détection + Tracking + Comptage + Anomalies

**Étapes:**
1. Auto-annotation (si besoin)
2. Entraînement YOLO
3. Processing vidéo complet

---

## 📦 CELL 1: Installation dépendances

In [2]:
# Installation une seule fois
!pip install ultralytics opencv-python pandas scikit-image pyyaml -q
print("✅ Dépendances installées!")

✅ Dépendances installées!


## 🔧 CELL 2: Imports

In [3]:
import cv2
import numpy as np
import os
import yaml
import pandas as pd
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print("✅ Tous les imports OK!")

✅ Tous les imports OK!


## 📁 CELL 3: Configuration chemins

In [4]:
# À ADAPTER SELON TON CHEMIN!
DATASET_ROOT = "dataset"  # Dossier avec images/ et labels/
VIDEO_PATH = "video.mp4"   # Ta vidéo
OUTPUT_DIR = "results"
MODEL_PATH = "runs/detect/savon_detector/weights/best.pt"
COUNTING_LINE_X = 0.85  # 85% à droite (sortie)

# Créer dossiers
Path(OUTPUT_DIR).mkdir(exist_ok=True)
Path(f"{DATASET_ROOT}/labels").mkdir(parents=True, exist_ok=True)

print(f"✅ Config:")
print(f"   Dataset: {DATASET_ROOT}")
print(f"   Vidéo: {VIDEO_PATH}")
print(f"   Output: {OUTPUT_DIR}")

✅ Config:
   Dataset: dataset
   Vidéo: video.mp4
   Output: results


---
# 🔍 ÉTAPE 1: AUTO-ANNOTATION (si besoin)
## Exécuter si tes 70 images ne sont PAS encore annotées

## CELL 4: Fonction auto-annotation par couleur

In [5]:
def detect_savons_color(img_path):
    """Détecte savons par plage couleur HSV"""
    img = cv2.imread(img_path)
    if img is None:
        return []
    
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Range pour savons marron/beige (AJUSTER si besoin!)
    lower_brown = np.array([10, 60, 50])
    upper_brown = np.array([25, 255, 200])
    
    mask = cv2.inRange(hsv, lower_brown, upper_brown)
    
    # Morphologie
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    boxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 300 < area < 50000:
            x, y, bw, bh = cv2.boundingRect(cnt)
            if bw > 0 and bh > 0:
                ratio = max(bw, bh) / min(bw, bh)
                if ratio < 5:
                    boxes.append((x, y, x + bw, y + bh))
    
    return boxes

def boxes_to_yolo(boxes, img_width, img_height):
    """Convertit boxes en format YOLO"""
    yolo_lines = []
    for x1, y1, x2, y2 in boxes:
        cx = ((x1 + x2) / 2) / img_width
        cy = ((y1 + y2) / 2) / img_height
        bw = (x2 - x1) / img_width
        bh = (y2 - y1) / img_height
        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    return yolo_lines

print("✅ Fonctions auto-annotation OK!")

✅ Fonctions auto-annotation OK!


## CELL 5: Exécuter auto-annotation

In [7]:
# AUTO-ANNOTATION
images_dir = os.path.join(DATASET_ROOT, "images")
labels_dir = os.path.join(DATASET_ROOT, "labels")

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
print(f"🔍 {len(image_files)} images trouvées\n")

total_boxes = 0
for img_file in image_files:
    img_path = os.path.join(images_dir, img_file)
    
    # Détection
    boxes = detect_savons_color(img_path)
    
    # Conversion YOLO
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    yolo_lines = boxes_to_yolo(boxes, w, h)
    
    # Écrire fichier .txt
    txt_filename = img_file.rsplit('.', 1)[0] + '.txt'
    txt_path = os.path.join(labels_dir, txt_filename)
    
    with open(txt_path, 'w') as f:
        for line in yolo_lines:
            f.write(line + '\n')
    
    total_boxes += len(boxes)
    status = "✓" if len(boxes) > 0 else "⚠️"
    if len(image_files) <= 10 or len(image_files) % 7 == 0:
        print(f"{status} {img_file}: {len(boxes)} box(es)")

print(f"\n✅ Auto-annotation terminée!")
print(f"   Total boxes: {total_boxes}")
print(f"   Moyenne par image: {total_boxes/len(image_files):.1f}")

🔍 70 images trouvées

✓ savon_0000.jpg: 3 box(es)
✓ savon_0001.jpg: 3 box(es)
✓ savon_0002.jpg: 3 box(es)
✓ savon_0003.jpg: 2 box(es)
✓ savon_0004.jpg: 2 box(es)
✓ savon_0005.jpg: 3 box(es)
✓ savon_0006.jpg: 2 box(es)
⚠️ savon_0007.jpg: 0 box(es)
✓ savon_0008.jpg: 1 box(es)
✓ savon_0009.jpg: 3 box(es)
✓ savon_0010.jpg: 2 box(es)
✓ savon_0011.jpg: 3 box(es)
✓ savon_0012.jpg: 2 box(es)
✓ savon_0013.jpg: 1 box(es)
✓ savon_0014.jpg: 2 box(es)
✓ savon_0015.jpg: 3 box(es)
✓ savon_0016.jpg: 3 box(es)
✓ savon_0017.jpg: 2 box(es)
✓ savon_0018.jpg: 1 box(es)
⚠️ savon_0019.jpg: 0 box(es)
✓ savon_0020.jpg: 1 box(es)
✓ savon_0021.jpg: 3 box(es)
✓ savon_0022.jpg: 2 box(es)
✓ savon_0023.jpg: 1 box(es)
⚠️ savon_0024.jpg: 0 box(es)
✓ savon_0025.jpg: 1 box(es)
✓ savon_0026.jpg: 3 box(es)
⚠️ savon_0027.jpg: 0 box(es)
✓ savon_0028.jpg: 2 box(es)
✓ savon_0029.jpg: 1 box(es)
✓ savon_0030.jpg: 3 box(es)
✓ savon_0031.jpg: 4 box(es)
✓ savon_0032.jpg: 4 box(es)
✓ savon_0033.jpg: 1 box(es)
✓ savon_0034.jpg: 1 bo

---
# 🤖 ÉTAPE 2: ENTRAÎNEMENT YOLO

## CELL 6: Créer data.yaml

In [6]:
# Créer data.yaml pour YOLO
data_yaml = {
    'path': os.path.abspath(DATASET_ROOT),
    'train': 'images',
    'val': 'images',
    'nc': 1,  # 1 classe: savon
    'names': ['savon']
}

yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ data.yaml créé: {yaml_path}")
print(f"   Path: {data_yaml['path']}")
print(f"   Classes: {data_yaml['names']}")

✅ data.yaml créé: dataset\data.yaml
   Path: C:\Users\soukaina\dataset
   Classes: ['savon']


## CELL 7: Entraîner YOLO (⏱️ 10-15 min)

In [ ]:
import cv2
import os
import pandas as pd
from ultralytics import YOLO
from collections import defaultdict

# ============ CONFIG ============
VIDEO_PATH = r"C:\Users\soukaina\Desktop\Stage huile\savons.mp4"
OUTPUT_DIR = r"C:\Users\soukaina\Desktop\Stage huile\results"
COUNTING_LINE_X = 0.85

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Vérifier vidéo existe
if not os.path.exists(VIDEO_PATH):
    print(f"❌ Vidéo introuvable: {VIDEO_PATH}")
    exit()

print(f"✅ Vidéo trouvée!")

# ============ MODÈLE PRÉ-ENTRAÎNÉ ============
print("🤖 Chargement modèle...")
model = YOLO('yolo11n.pt')  # Pré-entraîné, pas besoin training!

# ============ VIDÉO ============
print(f"\n📹 Ouverture vidéo...")
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"   Résolution: {w}x{h}")
print(f"   FPS: {fps}")
print(f"   Frames: {total_frames}")

# Video writer (résolution réduite = moins d'espace)
output_w, output_h = w // 2, h // 2
output_video = os.path.join(OUTPUT_DIR, "output_annotated.mp4")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (output_w, output_h))

# ============ PROCESSING ============
print(f"\n⚙️  Processing en cours...\n")

counted_ids = set()
total_count = 0
counting_line_pos = int(output_w * COUNTING_LINE_X)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f"   Frame {frame_idx}/{total_frames}")
    
    # Réduire résolution
    frame = cv2.resize(frame, (output_w, output_h))
    
    # DÉTECTION
    results = model(frame, conf=0.4, verbose=False)
    
    # COMPTAGE + DESSINAGE
    for det in results[0].boxes:
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        conf = float(det.conf[0])
        
        cx = (x1 + x2) // 2
        obj_id = hash((x1, y1, x2, y2)) % 10000
        
        # Comptage à droite
        if cx > counting_line_pos and obj_id not in counted_ids:
            counted_ids.add(obj_id)
            total_count += 1
            color = (0, 255, 0)
            status = "✓"
        else:
            color = (0, 165, 255)
            status = "→"
        
        # Dessiner
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        label = f"ID:{obj_id%100} {status}"
        cv2.putText(frame, label, (x1, max(15, y1-5)), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Ligne comptage
    cv2.line(frame, (counting_line_pos, 0), (counting_line_pos, output_h), (0, 255, 255), 3)
    cv2.putText(frame, f"SORTIE | Total: {total_count}", 
               (max(10, counting_line_pos-100), 40), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    
    out.write(frame)

cap.release()
out.release()

# ============ RÉSULTATS ============
print(f"\n✅ Processing terminé!")
print(f"✅ Total comptés: {total_count}")
print(f"✅ Vidéo: {output_video}")

# Rapport
report_data = {
    'Métrique': [
        'Total comptés',
        'Durée vidéo (sec)',
        'Savons/sec'
    ],
    'Valeur': [
        total_count,
        f"{total_frames/fps:.1f}",
        f"{total_count/(total_frames/fps):.2f}" if total_frames > 0 else "0"
    ]
}

df = pd.DataFrame(report_data)
csv_path = os.path.join(OUTPUT_DIR, "report.csv")
df.to_csv(csv_path, index=False)

print(f"\n📊 Rapport CSV: {csv_path}")
print("\n" + "="*50)
print(df.to_string(index=False))
print("="*50)

✅ Vidéo trouvée!
🤖 Chargement modèle...

📹 Ouverture vidéo...
   Résolution: 478x850
   FPS: 29.9768148816581
   Frames: 2098

⚙️  Processing en cours...

   Frame 100/2098
   Frame 200/2098
   Frame 300/2098
   Frame 400/2098
   Frame 900/2098
   Frame 1000/2098
   Frame 1100/2098
   Frame 1200/2098
   Frame 1300/2098
   Frame 1400/2098
   Frame 1500/2098
   Frame 1600/2098
   Frame 1700/2098
   Frame 1800/2098


In [22]:
import shutil
import os
import subprocess

# Check disk space
total, used, free = shutil.disk_usage("C:/")
print(f"Free space: {free / (1024**3):.2f} GB")

# Delete old runs/cache
if os.path.exists("runs"):
    shutil.rmtree("runs")
    print("✅ runs/ deleted")

# Windows: Empty temp
os.system("del /Q %temp%\\*")
print("✅ Temp cleared")

Free space: 0.00 GB
✅ runs/ deleted
✅ Temp cleared


In [23]:
labels_path = os.path.join(DATASET_ROOT, 'labels')
label_files = [f for f in os.listdir(labels_path) if f.endswith('.txt')]
print(f"Label files: {len(label_files)}")

# Check if any labels are EMPTY
for lbl in label_files[:3]:
    with open(os.path.join(labels_path, lbl), 'r') as f:
        content = f.read()
        if not content.strip():
            print(f"❌ EMPTY: {lbl}")
        else:
            print(f"✅ OK: {lbl} - {content[:50]}")

Label files: 0


In [24]:
import os
import cv2
import numpy as np
from pathlib import Path

DATASET_ROOT = r"C:\Users\soukaina\Desktop\Stage huile\anaconda_projects\db\dataset"
images_path = os.path.join(DATASET_ROOT, 'images')
labels_path = os.path.join(DATASET_ROOT, 'labels')

Path(labels_path).mkdir(parents=True, exist_ok=True)

# ============================================
# AUTO-ANNOTATE (HSV color detection)
# ============================================
print("🚀 Auto-annotating images...\n")

for img_file in os.listdir(images_path):
    if not img_file.endswith(('.jpg', '.png')):
        continue
    
    img_path = os.path.join(images_path, img_file)
    img = cv2.imread(img_path)
    
    if img is None:
        print(f"❌ Failed to read: {img_file}")
        continue
    
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Detect soap (brownish/tan color - adjust based on your soaps!)
    lower = np.array([10, 50, 50])
    upper = np.array([30, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    
    # Morphology
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Generate YOLO format labels
    label_file = os.path.join(labels_path, img_file.replace('.jpg', '.txt').replace('.png', '.txt'))
    
    with open(label_file, 'w') as f:
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 100:  # Skip tiny noise
                continue
            
            x, y, bw, bh = cv2.boundingRect(cnt)
            
            # YOLO format: class_id center_x center_y width height (normalized 0-1)
            cx = (x + bw/2) / w
            cy = (y + bh/2) / h
            bw_norm = bw / w
            bh_norm = bh / h
            
            f.write(f"0 {cx:.4f} {cy:.4f} {bw_norm:.4f} {bh_norm:.4f}\n")
    
    print(f"✅ {img_file}")

print(f"\n✅ Auto-annotation done!")
print(f"Labels saved to: {labels_path}")

🚀 Auto-annotating images...

✅ savon_0000.jpg
✅ savon_0001.jpg
✅ savon_0002.jpg
✅ savon_0003.jpg
✅ savon_0004.jpg
✅ savon_0005.jpg
✅ savon_0006.jpg
✅ savon_0007.jpg
✅ savon_0008.jpg
✅ savon_0009.jpg
✅ savon_0010.jpg
✅ savon_0011.jpg
✅ savon_0012.jpg
✅ savon_0013.jpg
✅ savon_0014.jpg
✅ savon_0015.jpg
✅ savon_0016.jpg
✅ savon_0017.jpg
✅ savon_0018.jpg
✅ savon_0019.jpg
✅ savon_0020.jpg
✅ savon_0021.jpg
✅ savon_0022.jpg
✅ savon_0023.jpg
✅ savon_0024.jpg
✅ savon_0025.jpg
✅ savon_0026.jpg
✅ savon_0027.jpg
✅ savon_0028.jpg
✅ savon_0029.jpg
✅ savon_0030.jpg
✅ savon_0031.jpg
✅ savon_0032.jpg
✅ savon_0033.jpg
✅ savon_0034.jpg
✅ savon_0035.jpg
✅ savon_0036.jpg
✅ savon_0037.jpg
✅ savon_0038.jpg
✅ savon_0039.jpg
✅ savon_0040.jpg
✅ savon_0041.jpg
✅ savon_0042.jpg
✅ savon_0043.jpg
✅ savon_0044.jpg
✅ savon_0045.jpg
✅ savon_0046.jpg
✅ savon_0047.jpg
✅ savon_0048.jpg
✅ savon_0049.jpg
✅ savon_0050.jpg
✅ savon_0051.jpg
✅ savon_0052.jpg
✅ savon_0053.jpg
✅ savon_0054.jpg
✅ savon_0055.jpg
✅ savon_0056.jpg
✅ 

In [25]:
label_files = [f for f in os.listdir(labels_path) if f.endswith('.txt')]
print(f"Label files created: {len(label_files)}")

for lbl in label_files[:3]:
    with open(os.path.join(labels_path, lbl), 'r') as f:
        print(f"{lbl}: {f.read()[:50]}")

Label files created: 70
savon_0000.txt: 0 0.9059 0.7676 0.1883 0.1071
0 0.9540 0.7106 0.03
savon_0001.txt: 0 0.9163 0.7888 0.0418 0.0388
0 0.5241 0.8694 0.64
savon_0002.txt: 0 0.7772 0.7835 0.0523 0.0235
0 0.7301 0.8476 0.53


In [28]:
import shutil

total, used, free = shutil.disk_usage("C:/")
print(f"C: Drive Status:")
print(f"  Total: {total / (1024**3):.1f} GB")
print(f"  Used: {used / (1024**3):.1f} GB")
print(f"  FREE: {free / (1024**3):.2f} GB")  # ← Probably ~0!

C: Drive Status:
  Total: 237.9 GB
  Used: 237.9 GB
  FREE: 0.00 GB


In [29]:
import os
os.system('rd /s /q "%systemdrive%\\$Recycle.bin"')
print("✅ Recycle bin emptied")

✅ Recycle bin emptied


In [33]:
import shutil
import os

# 1. DELETE EVERYTHING
if os.path.exists("runs"):
    shutil.rmtree("runs")
    print("✅ Deleted runs/")

# 2. CLEAN CACHE
os.system("del /Q %temp%\\* 2>nul")
print("✅ Temp cleaned")

# 3. CHECK SPACE
total, used, free = shutil.disk_usage("C:/")
print(f"\n📊 Free space: {free / (1024**3):.2f} GB")

# 4. TRAIN WITH MINIMAL CONFIG
from ultralytics import YOLO
import yaml

yaml_path = r"C:\Users\soukaina\Desktop\Stage huile\anaconda_projects\db\dataset\data.yaml"
model = YOLO('yolov8n.pt')  # ← nano model (smaller!)

results = model.train(
    data=yaml_path,
    epochs=10,        # ← MINIMAL
    imgsz=320,        # ← TINY
    batch=2,          # ← TINY
    patience=3,
    device='cpu',
    save=True,
    project='runs/detect',
    name='savon_detector',
    save_period=-1,   # ← NO intermediate saves!
    plots=False,      # ← NO plot images
    verbose=False
)

print("\n✅ Training done!")
print("📁 Best model: runs/detect/savon_detector/weights/best.pt")

✅ Deleted runs/
✅ Temp cleaned

📊 Free space: 4.02 GB
New https://pypi.org/project/ultralytics/8.4.142 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.41  Python-3.13.9 torch-2.11.0+cpu CPU (Intel Core i5-6300U 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\soukaina\Desktop\Stage huile\anaconda_projects\db\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

---
# 🎯 ÉTAPE 3: PROCESSING VIDÉO COMPLET
## Détection + Tracking + Comptage + Anomalies

## CELL 8: Classe Tracker simple

In [9]:
class SimpleTracker:
    def __init__(self, max_age=30):
        self.tracks = {}
        self.next_id = 0
        self.max_age = max_age
        self.frame_count = 0
        
    def update(self, detections):
        self.frame_count += 1
        
        # Centroïdes des détections
        centroids = []
        for det in detections:
            x1, y1, x2, y2 = det[:4]
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            centroids.append((cx, cy, det))
        
        # Matching
        matched_ids = {}
        for track_id, track_data in self.tracks.items():
            if track_data['age'] > self.max_age:
                continue
            
            best_dist = float('inf')
            best_idx = -1
            
            for idx, (cx, cy, det) in enumerate(centroids):
                if idx in matched_ids:
                    continue
                
                dist = np.sqrt((cx - track_data['cx'])**2 + (cy - track_data['cy'])**2)
                if dist < best_dist and dist < 50:
                    best_dist = dist
                    best_idx = idx
            
            if best_idx >= 0:
                matched_ids[best_idx] = track_id
                cx, cy, det = centroids[best_idx]
                self.tracks[track_id]['cx'] = cx
                self.tracks[track_id]['cy'] = cy
                self.tracks[track_id]['det'] = det
                self.tracks[track_id]['age'] = 0
        
        # Results
        results = []
        for track_id, track_data in self.tracks.items():
            if track_data['age'] <= self.max_age:
                det = track_data['det']
                results.append((*det[:4], track_id))
        
        # Nouvelles tracks
        for idx, (cx, cy, det) in enumerate(centroids):
            if idx not in matched_ids:
                self.tracks[self.next_id] = {'cx': cx, 'cy': cy, 'det': det, 'age': 0}
                results.append((*det[:4], self.next_id))
                self.next_id += 1
        
        # Age
        for track_id in self.tracks:
            self.tracks[track_id]['age'] += 1
        
        return results

print("✅ Classe Tracker OK!")

✅ Classe Tracker OK!


## CELL 9: Classe Anomaly Detector

In [10]:
class AnomalyDetector:
    def __init__(self):
        self.anomalies = defaultdict(list)
    
    def check_packaging_damaged(self, crop, threshold=0.3):
        """Packaging déchiré - variation texture"""
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        variance = laplacian.var()
        return variance > threshold * 1000
    
    def check_triangle_shape(self, crop):
        """Pas triangle - analyse contours"""
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            return False
        
        cnt = max(contours, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        
        # Triangle = 3 points
        return 2 <= len(approx) <= 4
    
    def check_alignment(self, crop, savon_mask, threshold=0.6):
        """Mal aligné - centrage dans box"""
        h, w = crop.shape[:2]
        moments = cv2.moments(savon_mask)
        
        if moments['m00'] == 0:
            return False
        
        cx = moments['m10'] / moments['m00']
        cy = moments['m01'] / moments['m00']
        center_x, center_y = w / 2, h / 2
        
        dist = np.sqrt((cx - center_x)**2 + (cy - center_y)**2)
        max_dist = (w + h) / 4 * threshold
        
        return dist < max_dist
    
    def check_packaging_present(self, crop, threshold=0.1):
        """Packaging absent - détecte pixels clairs"""
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        white_pixels = np.sum(gray > 200) / gray.size
        return white_pixels > threshold

print("✅ Classe AnomalyDetector OK!")

✅ Classe AnomalyDetector OK!


## CELL 10: MAIN PROCESSING VIDEO (⏱️ 5-10 min)

In [37]:
import os
import sys
import cv2
import numpy as np
import csv
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

PROJECT_DIR = r"C:\Users\soukaina\Desktop\Stage huile"
VIDEO_PATH = os.path.join(PROJECT_DIR, "savons.mp4")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "results")

# Créer répertoire output
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paramètres
COUNTING_LINE_X = 0.5  # Position ligne comptage (50% largeur vidéo)
CONFIDENCE_THRESHOLD = 0.5
MAX_TRACKING_DISTANCE = 100  # pixels

# ============================================================================
# GESTION MODÈLE YOLO
# ============================================================================

def load_yolo_model():
    """Charger le modèle YOLO avec fallback"""
    
    model_paths = [
        os.path.join(PROJECT_DIR, "runs", "detect", "savon_detector", "weights", "best.pt"),
        os.path.join(PROJECT_DIR, "best.pt"),
        os.path.join(PROJECT_DIR, "models", "best.pt"),
        None  # Fallback: modèle par défaut YOLOv8
    ]
    
    for model_path in model_paths:
        if model_path and os.path.exists(model_path):
            try:
                print(f"✓ Modèle trouvé: {model_path}")
                model = YOLO(model_path)
                return model
            except Exception as e:
                print(f"✗ Erreur chargement {model_path}: {e}")
                continue
    
    # Fallback: utiliser YOLOv8 nano pré-entraîné
    print("⚠️  Aucun modèle personnalisé trouvé.")
    print("📥 Utilisation du modèle YOLOv8 nano pré-entraîné...")
    model = YOLO('yolov8n.pt')
    print("✓ Modèle YOLOv8n chargé avec succès")
    return model

# ============================================================================
# TRACKER SIMPLE
# ============================================================================

class SimpleTracker:
    def __init__(self, max_distance=MAX_TRACKING_DISTANCE):
        self.tracks = {}
        self.next_id = 1
        self.max_distance = max_distance
    
    def update(self, detections):
        """Update tracks with new detections
        detections: list of (x1, y1, x2, y2, conf)
        """
        tracked = []
        
        if not detections:
            self.tracks = {}
            return tracked
        
        # Centroïdes détections
        detection_centers = []
        for x1, y1, x2, y2, conf in detections:
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            detection_centers.append((cx, cy, x1, y1, x2, y2, conf))
        
        # Matcher avec tracks existants
        matched_tracks = set()
        for track_id, (last_cx, last_cy) in list(self.tracks.items()):
            best_dist = float('inf')
            best_idx = -1
            
            for idx, (cx, cy, x1, y1, x2, y2, conf) in enumerate(detection_centers):
                dist = np.sqrt((cx - last_cx)**2 + (cy - last_cy)**2)
                if dist < best_dist and dist < self.max_distance:
                    best_dist = dist
                    best_idx = idx
            
            if best_idx >= 0:
                cx, cy, x1, y1, x2, y2, conf = detection_centers[best_idx]
                self.tracks[track_id] = (cx, cy)
                tracked.append((x1, y1, x2, y2, track_id))
                matched_tracks.add(best_idx)
        
        # Nouvelles détections
        for idx, (cx, cy, x1, y1, x2, y2, conf) in enumerate(detection_centers):
            if idx not in matched_tracks:
                track_id = self.next_id
                self.next_id += 1
                self.tracks[track_id] = (cx, cy)
                tracked.append((x1, y1, x2, y2, track_id))
        
        return tracked

# ============================================================================
# DÉTECTEUR D'ANOMALIES
# ============================================================================

class AnomalyDetector:
    def __init__(self):
        pass
    
    def check_packaging_damaged(self, crop):
        """Détecte emballage déchiré"""
        if crop.size == 0:
            return False
        
        # Convertir en HSV pour détecter les déformations
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        
        # Edges
        edges = cv2.Canny(gray, 50, 150)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Si trop de contours = potentiellement déchiré
        return len(contours) > 10
    
    def check_triangle_shape(self, crop):
        """Vérifie que c'est un triangle"""
        if crop.size == 0:
            return False
        
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if len(contours) == 0:
            return False
        
        largest_contour = max(contours, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest_contour, True)
        approx = cv2.approxPolyDP(largest_contour, epsilon, True)
        
        # Triangle = 3 sommets
        return 3 <= len(approx) <= 4
    
    def check_alignment(self, crop, mask):
        """Vérifie l'alignement du savon"""
        if crop.size == 0:
            return True
        
        # Vérifier que le savon occupe une bonne portion de la zone
        filled_ratio = np.sum(mask > 0) / mask.size
        return 0.3 < filled_ratio < 0.95
    
    def check_packaging_present(self, crop):
        """Vérifie présence d'emballage"""
        if crop.size == 0:
            return False
        
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        return np.mean(gray) > 30  # Au minimum un peu de contenu

# ============================================================================
# TRAITEMENT VIDÉO
# ============================================================================

def process_video(video_path, model, output_dir):
    """Traiter la vidéo et retourner les résultats"""
    
    print(f"📹 Ouverture vidéo: {video_path}")
    
    if not os.path.exists(video_path):
        print(f"❌ Vidéo non trouvée: {video_path}")
        return None
    
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"❌ Impossible d'ouvrir la vidéo")
        return None
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"   Résolution: {w}x{h}")
    print(f"   FPS: {fps:.2f}")
    print(f"   Frames: {total_frames}")
    
    # Initialiser les composants
    tracker = SimpleTracker()
    anomaly_detector = AnomalyDetector()
    
    # Vidéo de sortie
    output_video = os.path.join(output_dir, "output_annotated.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (w, h))
    
    # Stats
    counted_ids = set()
    total_count = 0
    frame_results = []
    counting_line_pos = int(w * COUNTING_LINE_X)
    
    print(f"\n⚙️  Traitement vidéo en cours...\n")
    
    frame_idx = 0
    anomalies_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        if frame_idx % max(1, total_frames // 10) == 0:
            progress = (frame_idx / total_frames) * 100
            print(f"   Progression: {progress:.1f}% (Frame {frame_idx}/{total_frames})")
        
        # ====== DÉTECTION ======
        results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        detections = []
        
        for det in results[0].boxes:
            x1, y1, x2, y2 = map(int, det.xyxy[0])
            conf = float(det.conf[0])
            detections.append((x1, y1, x2, y2, conf))
        
        # ====== TRACKING ======
        tracked = tracker.update(detections)
        
        # ====== COMPTAGE + ANOMALIES ======
        frame_data = {
            'frame': frame_idx,
            'timestamp': frame_idx / fps if fps > 0 else 0,
            'detections': len(tracked),
            'anomalies': []
        }
        
        for x1, y1, x2, y2, track_id in tracked:
            # COMPTAGE
            cx = (x1 + x2) // 2
            if cx > counting_line_pos and track_id not in counted_ids:
                counted_ids.add(track_id)
                total_count += 1
                status = "✓"
            else:
                status = "→"
            
            # Extraction zone savon
            crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
            
            if crop.size > 0:
                # ====== DÉTECTION ANOMALIES ======
                anomalies_list = []
                
                if anomaly_detector.check_packaging_damaged(crop):
                    anomalies_list.append("PACKAGING_DÉCHIRÉ")
                
                if not anomaly_detector.check_triangle_shape(crop):
                    anomalies_list.append("PAS_TRIANGLE")
                
                gray_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                _, mask = cv2.threshold(gray_crop, 100, 255, cv2.THRESH_BINARY)
                if not anomaly_detector.check_alignment(crop, mask):
                    anomalies_list.append("MAL_ALIGNÉ")
                
                if not anomaly_detector.check_packaging_present(crop):
                    anomalies_list.append("PACKAGING_ABSENT")
                
                frame_data['anomalies'].append({
                    'track_id': track_id,
                    'anomalies': anomalies_list,
                    'status': status
                })
                
                if anomalies_list:
                    anomalies_count += 1
                
                # ====== DESSINER SUR FRAME ======
                color = (0, 255, 0) if not anomalies_list else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                label = f"ID:{track_id} {status}"
                if anomalies_list:
                    label += " ⚠️"
                cv2.putText(frame, label, (x1, y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        # Ligne de comptage
        cv2.line(frame, (counting_line_pos, 0), (counting_line_pos, h), (0, 255, 255), 2)
        cv2.putText(frame, f"SORTIE | Comptage: {total_count}", 
                   (counting_line_pos - 150, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        out.write(frame)
        frame_results.append(frame_data)
    
    cap.release()
    out.release()
    
    print(f"\n✅ Traitement terminé!")
    print(f"✅ Total savons comptés: {total_count}")
    print(f"✅ Anomalies détectées: {anomalies_count}")
    print(f"✅ Vidéo annotée: {output_video}")
    
    return {
        'output_video': output_video,
        'total_count': total_count,
        'anomalies_count': anomalies_count,
        'frame_results': frame_results
    }

# ============================================================================
# GÉNÉRATION RAPPORT CSV
# ============================================================================

def generate_csv_report(results, output_dir):
    """Générer rapport CSV détaillé"""
    
    if results is None:
        print("❌ Impossible de générer le rapport (aucun résultat)")
        return None
    
    csv_file = os.path.join(output_dir, "soap_counting_report.csv")
    
    with open(csv_file, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        
        # Headers
        writer.writerow(['RAPPORT COMPTAGE SAVONS'])
        writer.writerow(['Date', datetime.now().strftime("%Y-%m-%d %H:%M:%S")])
        writer.writerow(['Total Savons Comptés', results['total_count']])
        writer.writerow(['Anomalies Détectées', results['anomalies_count']])
        writer.writerow([])
        
        # Détails par frame
        writer.writerow(['Frame', 'Timestamp (s)', 'Détections', 'ID Track', 'Anomalies', 'Statut'])
        
        for frame_data in results['frame_results']:
            frame = frame_data['frame']
            timestamp = frame_data['timestamp']
            detections = frame_data['detections']
            
            if frame_data['anomalies']:
                for anom in frame_data['anomalies']:
                    track_id = anom['track_id']
                    anomalies = '; '.join(anom['anomalies']) if anom['anomalies'] else 'OK'
                    status = anom['status']
                    writer.writerow([frame, f"{timestamp:.2f}", detections, track_id, anomalies, status])
            else:
                writer.writerow([frame, f"{timestamp:.2f}", detections, '', 'Aucune', ''])
    
    print(f"✅ Rapport CSV généré: {csv_file}")
    return csv_file

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("🧼 SYSTÈME DE COMPTAGE ET DÉTECTION D'ANOMALIES - SAVONS")
    print("="*70)
    
    # Charger modèle
    print("\n🤖 Chargement du modèle...\n")
    model = load_yolo_model()
    
    # Traiter vidéo
    print()
    results = process_video(VIDEO_PATH, model, OUTPUT_DIR)
    
    if results:
        # Générer rapport
        print()
        csv_file = generate_csv_report(results, OUTPUT_DIR)
        
        print("\n" + "="*70)
        print("📊 RÉSUMÉ FINAL")
        print("="*70)
        print(f"Total savons comptés: {results['total_count']}")
        print(f"Anomalies détectées: {results['anomalies_count']}")
        print(f"Vidéo annotée: {results['output_video']}")
        print(f"Rapport CSV: {csv_file}")
        print("="*70)
    else:
        print("❌ Erreur lors du traitement de la vidéo")
        sys.exit(1)

🧼 SYSTÈME DE COMPTAGE ET DÉTECTION D'ANOMALIES - SAVONS

🤖 Chargement du modèle...

⚠️  Aucun modèle personnalisé trouvé.
📥 Utilisation du modèle YOLOv8 nano pré-entraîné...
✓ Modèle YOLOv8n chargé avec succès

📹 Ouverture vidéo: C:\Users\soukaina\Desktop\Stage huile\savons.mp4
   Résolution: 478x850
   FPS: 29.98
   Frames: 2098

⚙️  Traitement vidéo en cours...

   Progression: 10.0% (Frame 209/2098)
   Progression: 19.9% (Frame 418/2098)
   Progression: 29.9% (Frame 627/2098)
   Progression: 39.8% (Frame 836/2098)
WARNING NMS time limit 2.050s exceeded
   Progression: 49.8% (Frame 1045/2098)
   Progression: 69.7% (Frame 1463/2098)
   Progression: 79.7% (Frame 1672/2098)
   Progression: 99.6% (Frame 2090/2098)

✅ Traitement terminé!
✅ Total savons comptés: 16
✅ Anomalies détectées: 108
✅ Vidéo annotée: C:\Users\soukaina\Desktop\Stage huile\results\output_annotated.mp4

✅ Rapport CSV généré: C:\Users\soukaina\Desktop\Stage huile\results\soap_counting_report.csv

📊 RÉSUMÉ FINAL
Total s

## CELL 11: Générer RAPPORT CSV

In [39]:
# ============================================================================
# ANALYSER LES ANOMALIES
# ============================================================================

from collections import defaultdict
import pandas as pd
import os

# Vérifier que le traitement vidéo a bien été effectué
if results is None:
    raise RuntimeError(
        "Aucun résultat disponible. Exécute d'abord process_video()."
    )

# Récupérer les données depuis results
frame_results = results['frame_results']
total_count = results['total_count']

# Compteurs
anomaly_count = defaultdict(int)
savon_anomalies = defaultdict(set)

# Parcours de toutes les frames
for frame_data in frame_results:

    for anom in frame_data.get('anomalies', []):

        track_id = anom['track_id']

        for anom_type in anom.get('anomalies', []):

            anomaly_count[anom_type] += 1
            savon_anomalies[track_id].add(anom_type)


# ============================================================================
# RAPPORT FINAL
# ============================================================================

report_data = {
    'Métrique': [
        'Total comptés',
        'Savons avec anomalies',
        'Packaging déchiré',
        'Pas triangle',
        'Mal aligné',
        'Packaging absent'
    ],

    'Valeur': [
        total_count,
        len(savon_anomalies),
        anomaly_count['PACKAGING_DÉCHIRÉ'],
        anomaly_count['PAS_TRIANGLE'],
        anomaly_count['MAL_ALIGNÉ'],
        anomaly_count['PACKAGING_ABSENT']
    ]
}

df = pd.DataFrame(report_data)


# ============================================================================
# SAUVEGARDE CSV
# ============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = os.path.join(
    OUTPUT_DIR,
    "report.csv"
)

df.to_csv(
    csv_path,
    index=False,
    encoding='utf-8-sig'
)


# ============================================================================
# AFFICHAGE
# ============================================================================

print("\n" + "=" * 60)
print("📊 RAPPORT FINAL")
print("=" * 60)

print(df.to_string(index=False))

print("=" * 60)
print(f"\n💾 CSV sauvegardé : {csv_path}")


📊 RAPPORT FINAL
             Métrique  Valeur
        Total comptés      16
Savons avec anomalies      51
    Packaging déchiré     106
         Pas triangle      97
           Mal aligné      46
     Packaging absent       0

💾 CSV sauvegardé : C:\Users\soukaina\Desktop\Stage huile\results\report.csv


## CELL 12: Afficher résultats finaux

In [40]:
print(f"\n🎉 RÉSULTATS FINALS")
print(f"="*60)
print(f"\n📊 Statistiques comptage:")
print(f"   • Total savons comptés: {total_count}")
print(f"   • Durée vidéo: {total_frames/fps:.1f} sec")
print(f"   • Savons/sec: {total_count/(total_frames/fps):.2f}")

print(f"\n⚠️  Anomalies détectées:")
print(f"   • Savons avec problèmes: {len(savon_anomalies)}")
print(f"   • Packaging déchiré: {anomaly_count['PACKAGING_DÉCHIRÉ']}")
print(f"   • Pas triangle: {anomaly_count['PAS_TRIANGLE']}")
print(f"   • Mal aligné: {anomaly_count['MAL_ALIGNÉ']}")
print(f"   • Packaging absent: {anomaly_count['PACKAGING_ABSENT']}")

print(f"\n📁 Fichiers générés:")
print(f"   ✓ {output_video}")
print(f"   ✓ {csv_path}")
print(f"\n" + "="*60)


🎉 RÉSULTATS FINALS

📊 Statistiques comptage:
   • Total savons comptés: 16
   • Durée vidéo: 70.0 sec
   • Savons/sec: 0.23

⚠️  Anomalies détectées:
   • Savons avec problèmes: 51
   • Packaging déchiré: 106
   • Pas triangle: 97
   • Mal aligné: 46
   • Packaging absent: 0

📁 Fichiers générés:
   ✓ C:\Users\soukaina\Desktop\Stage huile\results\output_annotated.mp4
   ✓ C:\Users\soukaina\Desktop\Stage huile\results\report.csv



---
## 📝 NOTES IMPORTANTES

### Si auto-annotation ne détecte pas bien:
**CELL 5** → Ajuster les plages HSV:
```python
lower_brown = np.array([10, 60, 50])      # Essayer d'autres valeurs
upper_brown = np.array([25, 255, 200])    # Par ex: [15, 100, 70] et [30, 200, 180]
```

### Si erreur mémoire GPU pendant training:
**CELL 7** → Réduire batch size:
```python
batch=4  # Au lieu de 8
```

### Ajuster position ligne comptage:
**CELL 3** → Modifier:
```python
COUNTING_LINE_X = 0.85  # 0.5 = milieu, 1.0 = très à droite
```

### Pour voir la vidéo:
Accéder à: `results/output_annotated.mp4`